In [6]:
from pyspark.sql import SparkSession
import pandas as pd
from pymongo import MongoClient

print("Iniciando sesión de Apache Spark")

# 1. Crear sesión de Spark estándar (sin conectores externos complejos)
spark = SparkSession.builder \
    .appName("SteamInsight_Spark_Analysis") \
    .getOrCreate()

try:
    # 2. CARGA DE DATOS (Usamos la vía segura)
    print("Extrayendo datos de MongoDB")
    client = MongoClient('mongodb://mongodb:27017')
    db = client['steam_db']
    
    # Traemos las reseñas a un DataFrame de Pandas temporal
    data_mongo = list(db.reviews_raw.find({}, {"_id":0, "language":1, "voted_up":1, "playtime_hours":1}))
    df_pandas = pd.DataFrame(data_mongo)

    print("Convirtiendo datos a un DataFrame de Spark")
    # 3. PASAMOS EL TESTIGO A SPARK
    # Aquí es donde Spark toma el control de los datos
    df_spark = spark.createDataFrame(df_pandas)

    print("Realizando procesamiento distribuido con Spark SQL")
    
    # 4. ANÁLISIS TÍPICO DE BIG DATA
    # Calculamos estadísticas por idioma usando el motor de Spark
    analisis_spark = df_spark.groupBy("language") \
                             .agg({"playtime_hours": "avg", "language": "count"}) \
                             .withColumnRenamed("avg(playtime_hours)", "Media_Horas") \
                             .withColumnRenamed("count(language)", "Total_Reseñas") \
                             .orderBy("Total_Reseñas", ascending=False)

    print("\nRESULTADOS OBTENIDOS POR EL MOTOR SPARK:")
    analisis_spark.show(10)

    # Ejemplo de filtro rápido en Spark
    print("Mostrando solo idiomas con alta participación (Spark Filter):")
    analisis_spark.filter(col("Total_Reseñas") > 10).show()

except Exception as e:
    print(f"Error: {e}")

finally:
    # 5. Cerrar spark
    spark.stop()
    print("Sesión de Spark cerrada.")

Iniciando sesión de Apache Spark
Extrayendo datos de MongoDB
Convirtiendo datos a un DataFrame de Spark
Realizando procesamiento distribuido con Spark SQL

RESULTADOS OBTENIDOS POR EL MOTOR SPARK:
+---------+-------------+------------------+
| language|Total_Reseñas|       Media_Horas|
+---------+-------------+------------------+
|  english|        25649| 65.62853522554455|
|  russian|         5641| 38.61668143946112|
| schinese|         4434| 61.06481732070368|
|brazilian|         1949| 40.56716264751152|
|  spanish|         1891| 60.81015335801159|
|   german|         1481| 54.11897366644154|
|   french|         1157| 44.19766637856528|
|  koreana|         1028| 67.23657587548638|
|  turkish|          922| 44.58015184381777|
|   polish|          770|29.776883116883113|
+---------+-------------+------------------+
only showing top 10 rows

Mostrando solo idiomas con alta participación (Spark Filter):
+---------+-------------+------------------+
| language|Total_Reseñas|       Media_Ho